In [32]:
!pip install astroquery --quiet

In [33]:
import numpy as np
import pandas as pd
from astropy.time import Time
from astroquery.jplhorizons import Horizons

In [34]:
def mag_abs(mag, alpha, dist, r, G=0.15):
    """
    Calculate the absolute magnitude H using the asteroid H-G model.

    Parameters
    ----------
    mag : float or array-like
        Apparent magnitude.
    alpha : float or array-like
        Phase angle in degrees.
    dist : float or array-like
        Observer-object distance Delta in au.
    r : float or array-like
        Heliocentric distance in au.
    G : float
        Asteroid slope parameter (default 0.15).

    Returns
    -------
    H : float or array-like
        Absolute magnitude.
    """
    alpha = np.radians(alpha)

    phi1 = np.exp(-3.33 * np.tan(alpha / 2)**0.63)
    phi2 = np.exp(-1.87 * np.tan(alpha / 2)**1.22)

    phase = (1 - G) * phi1 + G * phi2

    H = mag - 5 * np.log10(r * dist) + 2.5 * np.log10(phase)

    return H

In [35]:
def query_dists_and_alpha(nombre_objeto, mjdobs):
    """
    Parameters
    ----------
    nombre_objeto : str
        Object designation recognized by JPL Horizons.
    mjdobs : float
        Observation time in Modified Julian Date.|

    Returns
    -------
    dist : float
        Observer-object distance in au.
    alpha : float
        Phase angle in degrees.
    r : float
        Heliocentric distance in au.
    """
    epoch_jd = Time(mjdobs, format='mjd', scale='utc').jd

    obj = Horizons(
        id=nombre_objeto,
        location='500',
        epochs=epoch_jd
    )

    eph = obj.ephemerides()

    dist = float(eph['delta'][0])
    alpha = float(eph['alpha'][0])
    r = float(eph['r'][0])

    return dist, alpha, r

In [36]:
def query_heliocentric_elements(nombre_objeto, mjdobs):
    """
    Parameters
    ----------
    nombre_objeto : str
        Object designation recognized by JPL Horizons.
    mjdobs : float
        Observation time in Modified Julian Date.

    Returns
    -------
    a : float
        Heliocentric semimajor axis in au.
    e : float
        Orbital eccentricity.
    i : float
        Heliocentric inclination in degrees.
    omega : float
        Argument of perihelion in degrees.
    Omega : float
        Longitude of ascending node in degrees.
    M : float
        Mean anomaly in degrees.
    """
    epoch_jd = Time(mjdobs, format='mjd', scale='utc').jd

    obj = Horizons(
        id=nombre_objeto,
        location='@sun',
        epochs=epoch_jd
    )

    elements = obj.elements()

    a = float(elements['a'][0])
    e = float(elements['e'][0])
    i = float(elements['incl'][0])
    omega = float(elements['w'][0])
    Omega = float(elements['Omega'][0])
    M = float(elements['M'][0])

    return a, e, i, omega, Omega, M

In [37]:
# this function calculates dist, alpha, r, a, e, i, omega, Omega, M

def fill_table(object_name, obs_time):

  dist, alpha, r = query_dists_and_alpha(object_name, obs_time)

  a, e, i, omega, Omega, M = query_heliocentric_elements(object_name, obs_time)

  return dist, alpha, r, a, e, i, omega, Omega, M

In [38]:
# load the data

df = pd.read_csv('/content/df_2025_MM82.csv')
df.shape

(464, 10)

In [39]:
df_2025_MM82 = df[df['provid']=='2025 MM82']
df_2025_MM82.shape

(464, 10)

In [40]:
object_names = df_2025_MM82['provid']
obs_times = df_2025_MM82['mjdobs']
mags = df_2025_MM82['mag']
bands = df_2025_MM82['band']

N = len(mags)

# example data
# object_names = ['2025 MY53', '2025 MY53', '2025 MY53']
# obs_times = [60904.315457, 60904.315457, 60904.315457]
# mags = [20,20.1,19.87]

# N = ...

In [41]:
# do the query and generate the master catalog

dist_ = []
alpha_ = []
r_ = []
a_ = []
e_ = []
i_ = []
omega_ = []
Omega_ = []
M_ = []

for index in range(N):

  print("Query number: "+str(index))

  dist, alpha, r, a, e, i, omega, Omega, M = fill_table(object_names.iloc[index], obs_times.iloc[index])

  dist_.append(dist)
  alpha_.append(alpha)
  r_.append(r)
  a_.append(a)
  e_.append(e)
  i_.append(i)
  omega_.append(omega)
  Omega_.append(Omega)
  M_.append(M)

df_master = pd.DataFrame({
    'object_names': object_names,
    'obs_times': obs_times,
    'mag': mags,
    'band': bands,
    'dist': dist_,
    'alpha': alpha_,
    'r': r_,
    'a': a_,
    'e': e_,
    'i': i_,
    'omega': omega_,
    'Omega': Omega_,
    'M': M_
})

Query number: 0
Query number: 1
Query number: 2
Query number: 3
Query number: 4
Query number: 5
Query number: 6
Query number: 7
Query number: 8
Query number: 9
Query number: 10
Query number: 11
Query number: 12
Query number: 13
Query number: 14
Query number: 15
Query number: 16
Query number: 17
Query number: 18
Query number: 19
Query number: 20
Query number: 21
Query number: 22
Query number: 23
Query number: 24
Query number: 25
Query number: 26
Query number: 27
Query number: 28
Query number: 29
Query number: 30
Query number: 31
Query number: 32
Query number: 33
Query number: 34
Query number: 35
Query number: 36
Query number: 37
Query number: 38
Query number: 39
Query number: 40
Query number: 41
Query number: 42
Query number: 43
Query number: 44
Query number: 45
Query number: 46
Query number: 47
Query number: 48
Query number: 49
Query number: 50
Query number: 51
Query number: 52
Query number: 53
Query number: 54
Query number: 55
Query number: 56
Query number: 57
Query number: 58
Query n

In [42]:
# add the H magnitudes to the master catalog

# calculate H
df_master['H'] = df_master.apply(
    lambda row: mag_abs(
        mag=row['mag'],
        alpha=row['alpha'],
        dist=row['dist'],
        r=row['r']
    ),
    axis=1
)

# divide in each band
bands_ = ['Li', 'Lg', 'Lz', 'Lr', 'Ly', 'Lu']
for band in bands_:
    df_master[f'H_{band}'] = df_master['H'].where(df_master['band'] == band)

In [43]:
# show the master catalog

df_master

,object_names,obs_times,mag,band,dist,alpha,r,a,e,i,omega,Omega,M,H,H_Li,H_Lg,H_Lz,H_Lr,H_Ly,H_Lu
0,2025 MM82,60793.135111,23.575,g,1.790324,12.7507,2.672010,2.888403,0.180803,7.247659,51.795207,67.921854,56.094741,19.423252,NaN,NaN,NaN,NaN,NaN,NaN
1,2025 MM82,60793.136087,23.404,g,1.790332,12.7510,2.672012,2.888403,0.180803,7.247659,51.795207,67.921854,56.094937,19.252229,NaN,NaN,NaN,NaN,NaN,NaN
2,2025 MM82,60793.136568,23.233,g,1.790337,12.7511,2.672013,2.888403,0.180803,7.247659,51.795207,67.921854,56.095033,19.081220,NaN,NaN,NaN,NaN,NaN,NaN
3,2025 MM82,60793.137540,23.318,g,1.790345,12.7515,2.672014,2.888403,0.180803,7.247659,51.795208,67.921854,56.095228,19.166193,NaN,NaN,NaN,NaN,NaN,NaN
4,2025 MM82,60793.138516,23.271,g,1.790354,12.7518,2.672016,2.888403,0.180803,7.247659,51.795208,67.921854,56.095424,19.119170,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459,2025 MM82,60799.098871,22.624,Lr,1.846823,14.5417,2.682779,2.888413,0.180802,7.247654,51.796472,67.921907,57.291026,18.331486,NaN,NaN,NaN,18.331486,NaN,NaN
460,2025 MM82,60798.091180,22.985,Lr,1.836801,14.2535,2.680955,2.888411,0.180803,7.247655,51.796263,67.921898,57.088888,18.715974,NaN,NaN,NaN,18.715974,NaN,NaN
461,2025 MM82,60795.157420,23.577,Lg,1.808707,13.3805,2.675655,2.888406,0.180803,7.247657,51.795644,67.921872,56.500396,19.377066,NaN,19.377066,NaN,NaN,NaN,NaN
462,2025 MM82,60795.155432,23.603,Lg,1.808688,13.3799,2.675651,2.888406,0.180803,7.247657,51.795643,67.921872,56.499998,19.403113,NaN,19.403113,NaN,NaN,NaN,NaN


In [45]:
df_master.to_csv('df_2025_MM82_master.csv', index=False)